**[NOTES — draft, for James to rewrite]**

This notebook compares visits flagged by a single algorithm with visits flagged by more than one (reviewer comments R1.11 and R1.13). Changes since the previous version:

- The cohort is now the analysis sample as the main pipeline defines it: every eligible visit (no missing acuity, age or race). The one-visit-per-patient restriction that used to be applied here is gone — that restriction now lives only in the sensitivity notebook `R2_visits.ipynb`.
- A check cell after the groups locks this notebook's counts to the main pipeline's numbers (flagged 209,429; single 153,032; multiple 56,397; none 208,294).
- Table 1's difference rows use patient-clustered risk/mean differences (GEE with robust sandwich SEs, from `ci_helpers.R`) instead of score intervals, chi-square and Welch tests, because patients can contribute more than one visit. Test columns show Wald z.
- Tables 2-3 are counts and percentages; their code is unchanged, their numbers move with the cohort. The Venn diagram that used to be Figure 1 here has moved to `3_descriptive.ipynb`, where it is Figure S1 (`figure_S1.*`).
- A title cell with a TOC (Data / Groups / Check / Tables 1-3) is still needed.

In [1]:
if(!require("pacman")) install.packages("pacman")
pacman::p_load(arrow, geepack, ggsci, gt, gtsummary, knitr, tidyverse)

# cluster-aware CI helpers shared by the analysis notebooks
source("ci_helpers.R")

Loading required package: pacman



# Data

In [2]:
ed_data <- read_parquet(
    "../MIMICED/ed_join_cc.parquet",
    col_select = c(subject_id, stay_id, intime, acuity, age, gender, race, pain,
                   disposition, length_of_stay, icd_chapter, nyu, res, acuity_out)
  ) |>
  left_join(
    read_parquet("../MIMICED/chiefcomp_clustered.parquet", col_select = c(stay_id, cluster)),
    by = "stay_id"
  ) |>
  mutate(
    combined_group = paste0(nyu, res, acuity_out)
  )

nrow(ed_data)

[1] 424707

# Groups

In [3]:
eligible <- ed_data |>
  filter(!is.na(acuity) & !is.na(age) & !is.na(race))

analysis <- eligible |>
  mutate(
    n_algorithms = nyu + res + acuity_out,
    group = factor(
      case_when(
        n_algorithms == 1 ~ "Single",
        n_algorithms >= 2 ~ "Multiple",
        TRUE ~ "None"
      ),
      levels = c("Single", "Multiple", "None")
    )
  )

flagged <- analysis |>
  filter(group != "None") |>
  mutate(group = fct_drop(group))

cell_labels <- c(
  "100" = "Diagnosis only",
  "010" = "Resources only",
  "001" = "Triage-Acuity only",
  "110" = "Diagnosis + Resources",
  "101" = "Diagnosis + Triage-Acuity",
  "011" = "Resources + Triage-Acuity",
  "111" = "All three"
)

n <- list(
  visits    = nrow(ed_data),
  eligible  = nrow(eligible),
  analysis  = nrow(analysis),
  flagged   = nrow(flagged),
  single    = sum(flagged$group == "Single"),
  multiple  = sum(flagged$group == "Multiple"),
  none      = sum(analysis$group == "None")
)

str(n)

List of 7
 $ visits  : int 424707
 $ eligible: int 417723
 $ analysis: int 417723
 $ flagged : int 209429
 $ single  : int 153032
 $ multiple: int 56397
 $ none    : int 208294


## Check

In [4]:
checks <- tibble(
  check = c("Visits with a discharge diagnosis", "Analysis sample",
            "Flagged by at least one algorithm", "Single algorithm", "Multiple algorithms",
            "Not identified by any algorithm"),
  observed = c(n$visits, n$analysis, n$flagged, n$single, n$multiple, n$none),
  expected = c(424707L, 417723L, 209429L, 153032L, 56397L, 208294L)
) |>
  mutate(result = if_else(observed == expected, "PASS", "FAIL"))

checks |>
  print(n = 10)

stopifnot(all(checks$result == "PASS"))
stopifnot(n$single + n$multiple == n$flagged)

# A tibble: 6 × 4
  check                             observed expected result
  <chr>                                <int>    <int> <chr> 
1 Visits with a discharge diagnosis   424707   424707 PASS  
2 Analysis sample                     417723   417723 PASS  
3 Flagged by at least one algorithm   209429   209429 PASS  
4 Single algorithm                    153032   153032 PASS  
5 Multiple algorithms                  56397    56397 PASS  
6 Not identified by any algorithm     208294   208294 PASS  


# Variables

In [5]:
table_vars <- function(data) {
  data |>
    transmute(
      group,
      subject_id,
      age,
      gender = case_when(
        gender == "M" ~ "Male",
        gender == "F" ~ "Female"
      ),
      race = case_when(
        str_detect(race, "WHITE") ~ "White",
        str_detect(race, "BLACK") ~ "Black",
        str_detect(race, "HISPANIC") ~ "Hispanic/Latino",
        str_detect(race, "ASIAN") ~ "Asian",
        race == "OTHER" ~ "Other",
        race == "UNKNOWN" ~ "Unknown",
        race == "UNABLE TO OBTAIN" ~ "Unable to obtain",
        race == "PATIENT DECLINED TO ANSWER" ~ "Declined to answer",
        race == "PORTUGUESE" ~ "Portuguese",
        race == "SOUTH AMERICAN" ~ "South American",
        race == "AMERICAN INDIAN/ALASKA NATIVE" ~ "American Indian/Alaska Native",
        race == "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER" ~ "Native Hawaiian/Pacific Islander",
        race == "MULTIPLE RACE/ETHNICITY" ~ "Multiple race/ethnicity",
        TRUE ~ str_to_sentence(race)
      ),
      acuity = as.integer(acuity),
      pain = as.integer(pain),
      length_of_stay,
      disposition = case_when(
        disposition == "HOME" ~ "Home",
        disposition == "ADMITTED" ~ "Admitted",
        disposition == "EXPIRED" ~ "Died in the ED",
        TRUE ~ "Other"
      ),
      cluster = cluster_names[as.character(cluster)],
      icd_chapter
    ) |>
    mutate(
      race        = fct_relevel(fct_infreq(race), "White", "Black", "Hispanic/Latino", "Asian"),
      cluster     = fct_infreq(cluster),
      icd_chapter = fct_infreq(icd_chapter)
    )
}

cluster_names <- c(
  "0"  = "Lower back pain",
  "1"  = "Influenza like illness",
  "2"  = "Vehicle collision + Altered mental status",
  "3"  = "Chest pain + dyspnea",
  "4"  = "Shortness breath",
  "5"  = "Abdomen pain",
  "6"  = "Leg + knee pain",
  "7"  = "Wound evaluation",
  "8"  = "Alcohol intoxication",
  "9"  = "Nausea vomiting",
  "10" = "Suicidal ideation",
  "11" = "Fall transfer"
)

comparison_table <- function(data) {
  # keep only the labels and digit rules for variables this table actually has
  in_data <- function(rules) rules[map_lgl(rules, \(f) all.vars(f)[1] %in% names(data))]

  data |>
    tbl_summary(
      by = group,
      include = -any_of("subject_id"),
      type = in_data(list(
        age ~ "continuous2",
        acuity ~ "continuous2",
        pain ~ "continuous2",
        length_of_stay ~ "continuous2"
      )),
      label = in_data(list(
        age ~ "Age (years)",
        gender ~ "Gender",
        race ~ "Self-reported race",
        acuity ~ "Triage acuity (ESI)",
        pain ~ "Pain score (0-10)",
        length_of_stay ~ "Length of stay (minutes)",
        disposition ~ "Discharge disposition",
        cluster ~ "Presenting complaint",
        icd_chapter ~ "Discharge diagnosis"
      )),
      statistic = list(
        all_continuous() ~ "{mean} ({sd})",
        all_categorical() ~ "{n} ({p}%)"
      ),
      digits = in_data(list(
        age ~ 1,
        acuity ~ 2,
        pain ~ 1,
        length_of_stay ~ 0
      )),
      missing = "ifany",
      missing_text = "Not recorded"
    ) |>
    bold_labels()
}

fmt_p <- function(p) if (p < 0.001) "p < 0.001" else sprintf("p = %.3f", p)

row_differences <- function(data) {
  fmt <- function(est, lo, hi, dp) {
    sprintf(paste0("%+.", dp, "f [%.", dp, "f to %.", dp, "f]"), est, lo, hi)
  }

  # cluster-robust difference between the two group levels (see ci_helpers.R);
  # the GEE coefficient is level 2 minus level 1, so the sign is flipped to
  # keep the table's first-minus-second convention
  diff_cells <- function(y, dp, scale = 1) {
    d <- gee_diff_ci(y, data$group, data$subject_id)
    list(
      difference = fmt(-d[["est"]] * scale, -d[["hi"]] * scale, -d[["lo"]] * scale, dp),
      test = sprintf("z = %.1f, %s", -d[["z"]], fmt_p(d[["p"]]))
    )
  }

  cont_rows <- function(v, dp) {
    cells <- diff_cells(data[[v]], dp)
    tibble(variable = v, label = "Mean (SD)", difference = cells$difference, test = cells$test)
  }

  miss_rows <- function(v) {
    cells <- diff_cells(is.na(data[[v]]), 1, scale = 100)
    tibble(variable = v, label = "Not recorded", difference = cells$difference, test = cells$test)
  }

  cat_rows <- function(v) {
    levels <- sort(unique(as.character(data[[v]][!is.na(data[[v]])])))
    map(levels, \(l) {
      cells <- diff_cells(data[[v]] == l, 1, scale = 100)
      tibble(variable = v, label = l, difference = cells$difference, test = cells$test)
    }) |>
      list_rbind()
  }

  continuous  <- c(age = 1, acuity = 2, pain = 1, length_of_stay = 0)
  categorical <- c("gender", "race", "disposition", "cluster", "icd_chapter")

  bind_rows(
    imap(continuous[names(continuous) %in% names(data)], \(dp, v) cont_rows(v, dp)) |> list_rbind(),
    map(names(continuous)[map_lgl(names(continuous), \(v) v %in% names(data) && anyNA(data[[v]]))], miss_rows) |> list_rbind(),
    map(categorical[categorical %in% names(data)], cat_rows) |> list_rbind()
  )
}

add_row_differences <- function(x, data) {
  x |>
    modify_table_body(\(.x) .x |>
      left_join(row_differences(data), by = c("variable", "label")) |>
      mutate(across(c(difference, test), \(v) replace_na(v, "")))) |>
    modify_header(difference = "**Difference (95% CI)**", test = "**Test**")
}

kable_summary <- function(x) {
  tbl <- x |>
    as_tibble() |>
    rename_with(\(nm) nm |> str_remove_all("\\*\\*") |> str_replace_all("\\s*\\n\\s*", ", ")) |>
    mutate(across(everything(), \(v) replace_na(str_remove_all(v, "^__|__$"), "")))

  kable(tbl, align = c("l", rep("r", ncol(tbl) - 1)))
}

# Table 1

In [6]:
table_1_data <- flagged |>
  table_vars() |>
  select(group, subject_id, age, gender, race, acuity, pain, length_of_stay, disposition)

table_1 <- table_1_data |>
  comparison_table() |>
  add_row_differences(table_1_data)

table_1 |>
  kable_summary()

table_1 |>
  as_gt() |>
  tab_source_note("95% CIs on differences are cluster-robust at the patient level (GEE with robust sandwich standard errors).") |>
  gtsave(filename = "R3_agreement_table_1.docx")



|Characteristic                   | Single, N = 153,032| Multiple, N = 56,397|    Difference (95% CI)|                  Test|
|:--------------------------------|-------------------:|--------------------:|----------------------:|---------------------:|
|Age (years)                      |                    |                     |                       |                      |
|Mean (SD)                        |         49.6 (20.2)|          45.5 (19.3)|      +4.1 [3.9 to 4.4]|   z = 36.3, p < 0.001|
|Gender                           |                    |                     |                       |                      |
|Female                           |        88,247 (58%)|         32,052 (57%)|      +0.8 [0.2 to 1.4]|    z = 2.7, p = 0.007|
|Male                             |        64,785 (42%)|         24,345 (43%)|    -0.8 [-1.4 to -0.2]|   z = -2.7, p = 0.007|
|Self-reported race               |                    |                     |                       |              

# Table 2

In [7]:
top_three <- function(data, var, by) {
  data |>
    count({{ by }}, value = {{ var }}) |>
    group_by({{ by }}) |>
    mutate(pct = 100 * n / sum(n)) |>
    slice_max(pct, n = 3, with_ties = FALSE) |>
    mutate(
      rank  = c("1st", "2nd", "3rd"),
      label = sprintf("%s (%.1f%%)", value, pct)
    ) |>
    ungroup() |>
    select({{ by }}, rank, label) |>
    pivot_wider(names_from = {{ by }}, values_from = label)
}

ranked_table <- function(data, by) {
  vars <- data |>
    table_vars() |>
    bind_cols(data |> select(by = {{ by }}))

  bind_rows(
    top_three(vars, cluster, by) |> mutate(section = "Common complaints (rank, %)"),
    top_three(vars, icd_chapter, by) |> mutate(section = "Common diagnoses (rank, %)")
  ) |>
    rename(Rank = rank) |>
    select(section, Rank, everything())
}

In [8]:
table_2 <- flagged |>
  ranked_table(group)

table_2 |>
  kable()

table_2 |>
  gt(groupname_col = "section") |>
  gtsave(filename = "R3_agreement_table_2.docx")



|section                     |Rank |Single                                            |Multiple                                       |
|:---------------------------|:----|:-------------------------------------------------|:----------------------------------------------|
|Common complaints (rank, %) |1st  |Wound evaluation (14.5%)                          |Wound evaluation (18.8%)                       |
|Common complaints (rank, %) |2nd  |Abdomen pain (13.9%)                              |Lower back pain (14.4%)                        |
|Common complaints (rank, %) |3rd  |Vehicle collision + Altered mental status (11.8%) |Leg + knee pain (13.5%)                        |
|Common diagnoses (rank, %)  |1st  |Ill-defined conditions (37.4%)                    |Ill-defined conditions (36.0%)                 |
|Common diagnoses (rank, %)  |2nd  |Injury and poisoning (11.8%)                      |Diseases of the musculoskeletal system (20.9%) |
|Common diagnoses (rank, %)  |3rd  |Diseases o

# Table 3

In [9]:
by_combination <- flagged |>
  mutate(combination = factor(cell_labels[combined_group], levels = cell_labels))

table_3 <- by_combination |>
  ranked_table(combination)

table_3 |>
  kable()

table_3 |>
  gt(groupname_col = "section") |>
  gtsave(filename = "R3_agreement_table_3.docx")



|section                     |Rank |Diagnosis only                                      |Resources only                 |Triage-Acuity only                            |Diagnosis + Resources                             |Diagnosis + Triage-Acuity                            |Resources + Triage-Acuity                                             |All three                                                              |
|:---------------------------|:----|:---------------------------------------------------|:------------------------------|:---------------------------------------------|:-------------------------------------------------|:----------------------------------------------------|:---------------------------------------------------------------------|:----------------------------------------------------------------------|
|Common complaints (rank, %) |1st  |Abdomen pain (19.1%)                                |Wound evaluation (21.6%)       |Wound evaluation (33.5%)                   